# 天気図パターン分類 - 手元で使う (v4)

VS Code でこのノートブックを開いて、上から順に実行してください。
**このリポジトリのフォルダの中だけで完結します**(GitHubにもColabにも繋ぎません)。

Colab版(`notebooks/predict.ipynb`)と**同じ関数**を呼んでいます
(`src/quicklook.py`)。片方だけ直して食い違うことがないよう、中身は1か所に
まとめてあります。

置き場所の決まり:

| | 場所 |
|---|---|
| 天気図(前処理後、2000〜2025年) | `data/processed/all/` |
| 重み | `weights/model.pt`・`weights/model_annot.pt` |
| H/L のテンプレート | `data/templates/` |
| 描き込んだ画像の書き出し先 | `reports/annotated/` |

**時代でフォルダを分けていません。**前処理がすべての天気図を 1453x1500 に
揃えるので、2000年でも2025年でも同じ設定で検出できます。

揃っているかは `python -m scripts.check_local_setup` で確かめられます。
足りないものがあれば、用意するコマンドまで出ます。

In [ ]:
# セットアップ(最初に1回だけ)
import sys
from pathlib import Path

# リポジトリのルートを import できるようにする。VS Code は
# ノートブックのある場所をカレントにすることがあるので、両方に対応する
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# **どのPythonで動いているかを必ず出す。**VS Code のカーネルは、
# ターミナルで有効化した仮想環境とは別のものが選ばれていることがある。
# その場合「ターミナルでは動くのにノートブックでは ModuleNotFoundError」
# という分かりにくい状態になる。
print(f"リポジトリ: {ROOT}")
print(f"Python    : {sys.executable}")

_missing = []
for _name in ("torch", "torchvision", "matplotlib", "cv2", "pandas", "PIL", "scipy"):
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

if _missing:
    print("\n★足りない道具: " + ", ".join(_missing))
    print("  上に出ている Python が、いつも使っている仮想環境と違っていませんか。")
    print("  違う場合: VS Code 右上の「カーネルの選択」で wpcvenv を選ぶ")
    print("  同じ場合: そのPythonに入れる ->")
    print(f'    & "{sys.executable}" -m pip install -r "{ROOT / "requirements.txt"}"')
else:
    %matplotlib inline

    from src.quicklook import (annotation_available, classify_and_show,
                               classify_date)

    ok, missing = annotation_available()
    print("注釈方式: " + ("使えます" if ok else f"使えません(足りない: {missing})"))
    print()
    print("揃っているかまとめて見るには、ターミナルで:")
    print("  python -m scripts.check_local_setup")


## 日付を指定して分類する

`data/processed/all` にある天気図を日付で引いて分類します。出るもの:

- 左: 検出した高低気圧の枠を描き込んだ天気図
- 右: 10ラベルの確信度
- 描き込んだ画像は `reports/annotated/` に保存されます

天気図は1日2回(00Z と 12Z)しかないので、`HOUR` は 0 か 12 だけです。

In [ ]:
# ここを書き換えて実行する
DATE = "2004-01-15"   # YYYY-MM-DD
HOUR = 0              # 0 か 12 のみ(天気図は1日2回)

# 表示するラベルのしきい値。None にすると校正ファイルのラベルごとの値を使う
THRESHOLD = None

classify_date(DATE, hour=HOUR, threshold=THRESHOLD, annotate=True)

## 何日かまとめて見る

日付を並べて回すだけです。1枚あたり数秒かかります。

In [ ]:
DATES = ["2004-01-15", "2004-01-16", "2004-01-17"]

for _d in DATES:
    classify_date(_d, hour=0, annotate=True)
    print("=" * 60)

## ファイルを直接指定して分類する

`data/processed/all` に入っていない天気図(切り出したばかりのものなど)を
見たいときは、パスで指定します。

In [ ]:
# ここを書き換えて実行する。ROOT からの相対で書けば、ノートブックを
# どこから開いても、フォルダごと移動しても動く
IMAGE = ROOT / "data" / "processed" / "all" / "Js_2023010100_page001.png"

# 表示するラベルのしきい値。None にすると校正ファイルのラベルごとの値を使う
THRESHOLD = None

# 検出した枠を描き込んでから分類する(左の絵で検出の当たり外れが見える)
USE_ANNOTATION = True

classify_and_show(IMAGE, threshold=THRESHOLD, annotate=USE_ANNOTATION)